In [1]:
from pathlib import Path
import pandas as pd


tropes_file = Path("d:/narana/data/tvtropes/tropes.csv")
lit_goodreads_file = Path("d:/narana/data/tvtropes/lit_goodreads_match.csv")
movies_file = Path("d:/narana/data/tvtropes/film_imdb_match.csv")

tropes_df = pd.read_csv(tropes_file)
tropes_df.dropna(subset=['Description'], inplace=True)
tropes_df.drop_duplicates(subset=['Description'], inplace=True)
lit_df = pd.read_csv(lit_goodreads_file)
movies_df = pd.read_csv(movies_file)

print(lit_df.columns.to_list())
print(movies_df.columns.to_list())
print(tropes_df.columns.to_list())



['Unnamed: 0', 'Title', 'Trope', 'Example', 'CleanTitle', 'author', 'verified_gender', 'title_id', 'trope_id']
['Unnamed: 0', 'Title', 'Trope', 'Example', 'CleanTitle', 'tconst', 'trope_id', 'title_id']
['Unnamed: 0', 'TropeID', 'Trope', 'Description']


In [2]:
# Count how many works match each trope in literature and movies
lit_trope_counts = lit_df.groupby('Trope').size().reset_index(name='lit_count')
movies_trope_counts = movies_df.groupby('Trope').size().reset_index(name='movies_count')

# Merge the counts with the tropes dataframe
tropes_df = tropes_df.merge(lit_trope_counts, on='Trope', how='left')
tropes_df = tropes_df.merge(movies_trope_counts, on='Trope', how='left')

# Fill NaN values with 0 (tropes that don't appear in either dataset)
tropes_df['lit_count'] = tropes_df['lit_count'].fillna(0).astype(int)
tropes_df['movies_count'] = tropes_df['movies_count'].fillna(0).astype(int)

# Add a total count column
tropes_df['total_count'] = tropes_df['lit_count'] + tropes_df['movies_count']

# Display some examples of tropes with their counts
print("\nTop 10 tropes by total count:")
print(tropes_df.sort_values('total_count', ascending=False).head(10)[['Trope', 'lit_count', 'movies_count', 'total_count']])

print("\nSome tropes that appear only in literature:")
lit_only = tropes_df[(tropes_df['lit_count'] > 0) & (tropes_df['movies_count'] == 0)]
print(lit_only.sample(min(5, len(lit_only)))[['Trope', 'lit_count']])

print("\nSome tropes that appear only in movies:")
movies_only = tropes_df[(tropes_df['lit_count'] == 0) & (tropes_df['movies_count'] > 0)]
print(movies_only.sample(min(5, len(movies_only)))[['Trope', 'movies_count']])



Top 10 tropes by total count:
                   Trope  lit_count  movies_count  total_count
20183           ShoutOut        688          1151         1839
3636         ChekhovsGun        467           716         1183
27656        HorrorFilms         13          1072         1085
13832     MeaningfulName        643           432         1075
8381       Foreshadowing        422           627         1049
2181              BigBad        367           635         1002
2330   BittersweetEnding        462           513          975
15961             OhCrap        220           719          939
5227      DeadpanSnarker        376           555          931
23966          TitleDrop        332           506          838

Some tropes that appear only in literature:
                           Trope  lit_count
10741               HungryWeapon          8
9177                GlurgeAddict          1
14919              MythologyPorn          1
24536  TwentyFourHourPartyPeople          1
15344      

In [3]:
print(len(tropes_df.loc[tropes_df['total_count'] > 30]))
tropes_df.loc[tropes_df['total_count'] > 30].to_csv("tropes.csv", index=False)


6278


In [4]:
lit_examples = pd.read_csv("d:/narana/data/tvtropes/lit_tropes.csv")
grouped_by_trope = lit_examples.groupby('Trope').size().reset_index(name='count').sort_values('count', ascending=False)
grouped_by_trope.head()

,Trope,count
20628,ShoutOut,1431
14293,MeaningfulName,1103
27122,YoungAdultLiterature,1058
2305,BigBad,953
2489,BittersweetEnding,788


In [5]:

tropes_with_10_to_20 = grouped_by_trope[(grouped_by_trope['count'] >= 10) & (grouped_by_trope['count'] <= 20)]


print(f"Number of tropes with 10-20 examples: {len(tropes_with_10_to_20)}")
print("\nSample of tropes with 10-20 examples:")
print(tropes_with_10_to_20.sample(min(10, len(tropes_with_10_to_20))))




Number of tropes with 10-20 examples: 6382

Sample of tropes with 10-20 examples:
                          Trope  count
25871            VisualInnuendo     11
5757             DesolationShot     10
154         AbandonedPlayground     11
7954            FantasticMetals     19
10216  HeWillNotCrySoICryForHim     11
15939          NoblewomansLaugh     16
25258           UndeathlyPallor     16
24833         TrappedInVillainy     17
20866     SituationalHandSwitch     11
21119           SmoochOfVictory     11


In [6]:

# Load the necessary data
import os

# Create a function to format trope data into the requested XML-like format
def format_trope_with_titles(trope_name, titles_data):
    output = f"<trope>{trope_name}<titles>"
    
    for title_name, example in titles_data:
        # Truncate example if it's too long (optional)
        output += f"<title name={title_name}>{example}</title>"
    
    output += "</titles></trope>"
    return output


def format_trope_description(trope_name, description):
    return f"<trope name={trope_name}>{description}</trope>"


trope_titles = {}
trope_examples = lit_examples[lit_examples['Trope'].isin(tropes_with_10_to_20['Trope'].unique())]
trope_titles = trope_examples.groupby('Trope').apply(lambda x: list(zip(x['Title'], x['Example']))).to_dict()
trope_descriptions = tropes_df[tropes_df['Trope'].isin(tropes_with_10_to_20['Trope'].unique())].set_index('Trope')['Description'].to_dict()

# output_dir = "trope_data"
# os.makedirs(output_dir, exist_ok=True)

# with open(os.path.join(output_dir, "tropes_with_titles.txt"), "w", encoding="utf-8") as f:
#     for trope, titles in trope_titles.items():
#         formatted_trope = format_trope_with_titles(trope, titles)
#         f.write(formatted_trope + "\n")


# trope_desc_items = list(trope_descriptions.items())
# num_items = len(trope_desc_items)
# items_per_file = (num_items + 9) // 10  # Ceiling division to ensure all items are covered

# for i in range(10):
#     start_idx = i * items_per_file
#     end_idx = min((i + 1) * items_per_file, num_items)
    
#     if start_idx >= num_items:
#         break
    
#     with open(os.path.join(output_dir, f"tropes_with_descriptions_split{i+1}.txt"), "w", encoding="utf-8") as f:
#         for trope, description in trope_desc_items[start_idx:end_idx]:
#             formatted_trope = format_trope_description(trope, description)
#             f.write(formatted_trope + "\n")


# print(f"Trope data has been saved to {os.path.join(output_dir, 'tropes_with_titles.txt')}")

# sample_trope = list(trope_titles.keys())[0]
# sample_formatted = format_trope_with_titles(sample_trope, trope_titles[sample_trope])
# print("\nSample of formatted trope data:")
# print(sample_formatted[:500] + "..." if len(sample_formatted) > 500 else sample_formatted)


C:\Users\Frantisek\AppData\Local\Temp\ipykernel_17052\787864831.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trope_titles = trope_examples.groupby('Trope').apply(lambda x: list(zip(x['Title'], x['Example']))).to_dict()


In [92]:
from tqdm import tqdm
import numpy as np
from scipy import sparse
from collections import defaultdict



# Group titles by their tropes
tropes_that_are_specific = pd.read_json("trope_data/combined.jsonl", lines=True)
tropes_that_are_specific = tropes_that_are_specific[tropes_that_are_specific["specificity"] > 0.0]
lit_examples = lit_examples[lit_examples["Trope"].isin(tropes_that_are_specific["trope"])]
lit_examples = pd.read_csv("d:/narana/data/tvtropes/lit_tropes.csv")
lit_examples["trope"] = lit_examples["Trope"].apply(lambda x: x.lower())
lit_examples.drop_duplicates(subset=["trope", "Example", "Title"], inplace=True)
titles = lit_examples.groupby("Title")["Trope"].apply(set).reset_index(name="tropes")
titles = titles[(titles["tropes"].apply(len) < 40) & (titles["tropes"].apply(len) > 10)]

# Create a mapping of trope names to indices
unique_tropes = list(set().union(*titles["tropes"]))
trope_to_idx = {trope: idx for idx, trope in enumerate(unique_tropes)}

# Create sparse vectors for each title
num_titles = len(titles)
num_tropes = len(unique_tropes)
rows = []
cols = []
data = []

for title_idx, row in enumerate(titles.itertuples()):
    for trope in row.tropes:
        trope_idx = trope_to_idx[trope]
        rows.append(title_idx)
        cols.append(trope_idx)
        data.append(1)

# Create sparse matrix where each row is a title and each column is a trope
title_trope_matrix = sparse.csr_matrix((data, (rows, cols)), shape=(num_titles, num_tropes))

# Compute dot product between all pairs of title vectors
# This gives us the number of overlapping tropes between each pair
overlap_matrix = title_trope_matrix.dot(title_trope_matrix.T).toarray()

print(overlap_matrix.shape)
# Extract the pairs and their overlap counts
title_pairs = []
for i in tqdm(range(num_titles)):
    title1 = titles.iloc[i]["Title"]
    tropes1 = titles.iloc[i]["tropes"]
    
    # Only look at upper triangle to avoid duplicates
    for j in range(i+1, num_titles):
        overlap_count = int(overlap_matrix[i, j])
        
        # Skip pairs with no overlap
        if overlap_count == 0:
            continue
            
        title2 = titles.iloc[j]["Title"]
        tropes2 = titles.iloc[j]["tropes"]
        
        # Find common tropes
        common_tropes = list(tropes1.intersection(tropes2))
        
        title_pairs.append({
            "title1": title1,
            "title2": title2,
            "common_tropes": common_tropes,
            "overlap_count": overlap_count
        })

# Convert to DataFrame for easier analysis
title_overlaps = pd.DataFrame(title_pairs)
if not title_overlaps.empty:
    # Sort by number of overlapping tropes in descending order
    title_overlaps = title_overlaps.sort_values("overlap_count", ascending=False)
 
title_overlaps.to_csv("title_overlaps.csv", index=False)
title_overlaps[title_overlaps["overlap_count"] > 10].sample(10)



(4003, 4003)


100%|██████████| 4003/4003 [00:40<00:00, 99.55it/s] 


,title1,title2,common_tropes,overlap_count
558787,OverSeaUnderStone,TheGreyKing,"[EvilDetectingAnimal, MissExposition, GottaCat...",11
612876,Sampaguita,Yarudora,"[RoadCone, CollectibleCardGame, NeatFreak, Bad...",15
723215,TheGooseGirl,TheLordOfLornAndTheFalseSteward,"[ExactWords, RagsToRoyalty, Genderflipped, Dis...",14
161243,CapORushes,Catskin,"[PrincessForADay, DancesAndBalls, RagsToRoyalt...",13
44892,AgeOfSteam,DeadIron,"[MosesInTheBulrushes, GreenThumb, Insubstantia...",14
86390,ArkRoyal,NamelessWar,"[SpaceFighter, StealthInSpace, BreakOutTheMuse...",11
410901,IfThisIsAMan,TheTruce,"[DeadlyEuphemism, InfantImmortality, AllJustAD...",22
358069,Greenwitch,TheGreyKing,"[EvilDetectingAnimal, MissExposition, GottaCat...",11
375735,HearTheWindSing,Pinball1973,"[CallForward, AgeAppropriateAngst, DrivenToSui...",13
5380,ACollegeOfMagics,ScholarlyMagics,"[OrientExpress, IllegalGuardian, SomethingOnly...",17


In [84]:
from ast import literal_eval
title_overlaps = pd.read_csv("title_overlaps.csv") 
title_overlaps = pd.read_csv("title_overlaps.csv", converters={"common_tropes": literal_eval})
# title_overlaps["common_tropes"] = title_overlaps["common_tropes"].apply(literal_eval)

EmptyDataError: No columns to parse from file

In [93]:
samples = title_overlaps[title_overlaps["overlap_count"] > 5].sample(5)
for index, row in samples.iterrows():
    
    tropes1 = set(titles[titles["Title"] == row["title1"]]["tropes"].item())
    tropes2 = set(titles[titles["Title"] == row["title2"]]["tropes"].item())
    
    # Get common tropes from the intersection of the two sets
    computed_common_tropes = sorted(list(tropes1.intersection(tropes2)))
    
    # Get common tropes from the dataframe
    stored_common_tropes = sorted(row["common_tropes"])
    
    # Check if they match
    match = computed_common_tropes == stored_common_tropes
    print(f"Common tropes match: {match}")
    
    if not match:
        print(f"Computed common tropes count: {len(computed_common_tropes)}")
        print(f"Stored common tropes count: {len(stored_common_tropes)}")
        print(f"Difference: {set(computed_common_tropes).symmetric_difference(set(stored_common_tropes))}")
            




Common tropes match: True
Common tropes match: True
Common tropes match: True
Common tropes match: True
Common tropes match: True


In [112]:
from sklearn.decomposition import NMF

n_topics = 500 

model = NMF(n_components=n_topics,
            init='nndsvda', 
            max_iter=500,   
            random_state=42,
            l1_ratio=0.0,  
            solver='cd')    


W = model.fit_transform(title_trope_matrix)
H = model.components_

W_df = pd.DataFrame(W, index=titles["Title"], columns=[f'Topic {i+1}' for i in range(n_topics)])
H_df = pd.DataFrame(H, index=[f'Topic {i+1}' for i in range(n_topics)], columns=unique_tropes)


In [115]:
n_top_books = 5
n_top_tropes = 4
for i in range(n_topics):
    topic_num = i + 1
   
   
    top_book_indices = np.argsort(W[:, i])[::-1]  # descending
    top_books = [titles["Title"].iloc[idx] for idx in top_book_indices[:n_top_books]]
    

    top_trope_indices = np.argsort(H[i, :])[::-1] #  descending

    tropes_sets = titles[titles["Title"].isin(top_books)]["tropes"].apply(set).to_list()
    intersection_tropes = set.intersection(*tropes_sets)
    if len(intersection_tropes) > n_top_tropes:
        print(f"--- Topic {topic_num} ---")
        print(f"Top {n_top_books} Books: {', '.join(top_books)}")
        print(f"Intersection of tropes: {intersection_tropes}")


--- Topic 9 ---
Top 5 Books: Momotaro, TheMagicTreehouse, TheMidnightFolk, Jabberwocky, KnightLifeSeries
Intersection of tropes: {'TheSpearOfDestiny', 'SpearOfDestiny', 'ShroudOfTurin', 'SevenLeagueBoots', 'HolyGrail', 'Mjolnir', 'TheArkOfTheCovenant', 'PublicDomainArtifact', 'TheLanceOfLonginus'}
--- Topic 21 ---
Top 5 Books: Pathfinder, HeistSociety, BlueRabbit, ThePathToWar, DanielX
Intersection of tropes: {'TheBigGuy', 'TheChick', 'TheLancer', 'FiveManBand', 'TheSmartGuy'}
--- Topic 206 ---
Top 5 Books: RadicalDreamers, HaveANiceDayATaleOfBloodAndSweatsocks, AssociatedSpace, JunipersKnot, AGiantSuckingSound
Intersection of tropes: {'TakeTheThirdOption', 'TookAThirdOption', 'TakeAThirdOption', 'ThirdOption', 'TakingAThirdOption', 'TakesAThirdOption'}
--- Topic 267 ---
Top 5 Books: MaxHavelaar, TheCandidatesBasedOnATrueCountry, SurfaceDetail, SingYouHome, TheEnglishDragon
Intersection of tropes: {'StrawPolitical', 'StrawLiberal', 'StrawConservative', 'StrawCharacter', 'StrawmanPoliti

In [133]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import community as community_louvain 



book_names = titles["Title"].to_list()
similarity_matrix = cosine_similarity(title_trope_matrix)

similarity_df = pd.DataFrame(similarity_matrix, index=book_names, columns=book_names)

G = nx.Graph()
G.add_nodes_from(book_names)


edge_threshold = 0.4 
num_books = len(book_names)
print(f"Building graph: Adding weighted edges for similarity > {edge_threshold}")
edges_added = 0
for i in range(num_books):
    for j in range(i + 1, num_books):
        similarity = similarity_matrix[i, j]
        if similarity > edge_threshold:
          
            G.add_edge(book_names[i], book_names[j], weight=similarity)
            edges_added += 1
print(f"Added {edges_added} edges to the graph.\n")

if edges_added > 0:
   
    print("Running Louvain Community Detection...")
    partition = community_louvain.best_partition(G, weight='weight', random_state=11)
    communities = {}
    for node, community_id in partition.items():
        if community_id not in communities:
            communities[community_id] = []
        communities[community_id].append(node)

    print("\nDetected Communities:")
    community_num = 1
    for community_id, members in communities.items():
        if len(members) > 2:
            print(f"Community {community_num} (ID {community_id}): {', '.join(members)}")
            members_tropes = set.intersection(*titles[titles["Title"].isin(members)]["tropes"].apply(set).to_list())
            print(f"Intersection of tropes: {members_tropes}")
            community_num += 1
else:
    print("No edges were added to the graph based on the threshold.")
    print("Cannot run community detection on a graph with no edges.")

print("\n" + "="*50 + "\n")

Building graph: Adding weighted edges for similarity > 0.4
Added 137 edges to the graph.

Running Louvain Community Detection...

Detected Communities:
Community 1 (ID 352): BludgeoningAngelDokurochan, DiscWorld, ExpectingSomeoneTaller, HighschoolDxD, HistoryOfTheKingsOfBritain, Jabberwocky, KnightLifeSeries, Magnus, Momotaro, NyarkoSan, ReMonster, TheConfessionsOfPeterCrossman, TheFourthRealm, TheHammerAndTheCross, TheKingsQuestCompanion, TheMagicTreehouse, TheMidnightFolk, ThePrometheanAge, TheTaleOfTheHeike, TheTreasureOfAlpheusWinterborn
Intersection of tropes: {'TheSpearOfDestiny', 'SpearOfDestiny', 'ShroudOfTurin', 'SevenLeagueBoots', 'HolyGrail', 'Mjolnir', 'TheArkOfTheCovenant', 'TheLanceOfLonginus'}
Community 2 (ID 430): CapORushes, Catskin, Donkeyskin, Tattercoats
Intersection of tropes: {'CharacterTitle', 'PersonWithTheClothing', 'TitleCharacter'}
Community 3 (ID 779): EastOfTheSunWestOfTheMoon, TheSevenRavens, TheSixSwans
Intersection of tropes: {'WickedStepfather', 'CurseE

In [1]:
lit_examples[lit_examples["Trope"] == "IJustWantToBeHuman"]

NameError: name 'lit_examples' is not defined